# hetero1d: real-world applications

**hetero1d** is a 1D semiconductor heterostructure solver (effective-mass envelope function,
BenDaniel-Duke finite differences). This notebook shows how it solves real design problems
from optoelectronics, electronics and quantum physics, each in a few lines of Python.

| # | Application | Real device / experiment |
|---|---|---|
| 1 | Laser wavelength design | 850 nm GaAs VCSEL / laser diode |
| 2 | Telecom laser design | 1.55 µm InGaAs/InP laser |
| 3 | Infrared photodetector design | 8–10 µm QWIP (thermal imaging) |
| 4 | Electro-absorption modulator | Quantum-confined Stark effect |
| 5 | Resonant tunnelling diode | Resonance energy and lifetime |
| 6 | Superlattice engineering | Miniband width control |
| 7 | High-electron-mobility transistor | 2DEG from modulation doping |
| 8 | Quantum Hall / magnetotransport | Landau levels, SdH oscillations, g-factor |
| 9 | Double quantum well in a tilted field | Tunnelling gap, camel-back dispersion |
| 10 | Exporting data | CSV for Origin / Excel / papers |

Project: https://github.com/erw-del/hetero1d  ·  Author: Ravi Kumar

## 0. Install and import

In [ ]:
!pip install -q git+https://github.com/erw-del/hetero1d.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hetero1d import Structure, Solver, material, lattice_matched, list_materials
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
list_materials()

## 1. Laser wavelength design: 850 nm GaAs quantum-well laser

**Problem:** VCSELs for data communication emit at **850 nm**. Which GaAs well width in
Al₀.₃Ga₀.₇As barriers gives this wavelength at room temperature?

**Method:** scan the well width, compute the lowest interband transition e0–hh0.

In [ ]:
Lws = np.arange(3, 21, 0.5)
lam = []
for Lw in Lws:
    s = Structure.single_well(well="GaAs", barrier=("AlGaAs", 0.3), Lw=Lw, Lb=25)
    sol = Solver(s, T=300).solve(n_e=3, n_h=3)
    lam.append(sol.interband()[0]["lam_nm"])
lam = np.array(lam)

Lw_850 = np.interp(850, lam, Lws)
print(f"Well width for 850 nm emission: {Lw_850:.2f} nm")

plt.plot(Lws, lam, "o-")
plt.axhline(850, color="r", ls="--"); plt.axvline(Lw_850, color="r", ls="--")
plt.xlabel("Well width (nm)"); plt.ylabel("Emission wavelength (nm)")
plt.title("GaAs/Al$_{0.3}$Ga$_{0.7}$As QW laser, 300 K"); plt.grid(alpha=.3); plt.show()

In [ ]:
# look at the designed laser well
s = Structure.single_well(well="GaAs", barrier=("AlGaAs", 0.3), Lw=round(Lw_850, 1), Lb=25)
sol = Solver(s, T=300).solve()
sol.report()
sol.plot_summary("laser_850nm.png")
from IPython.display import Image; Image("laser_850nm.png")

**Temperature drift.** Lasers heat up during operation. How much does the wavelength shift?

In [ ]:
Ts = np.arange(10, 401, 30)
lamT = [Solver(s, T=T).solve(3, 3).interband()[0]["lam_nm"] for T in Ts]
slope = np.polyfit(Ts[Ts > 200], np.array(lamT)[Ts > 200], 1)[0]
print(f"Wavelength drift near room temperature: {slope:.3f} nm/K")
plt.plot(Ts, lamT, "o-"); plt.xlabel("Temperature (K)"); plt.ylabel("Wavelength (nm)")
plt.title("Temperature tuning of the 850 nm laser well"); plt.grid(alpha=.3); plt.show()

## 2. Telecom laser: 1.55 µm InGaAs/InP

**Problem:** optical fibre has its lowest loss at **1550 nm**. Design an In$_x$Ga$_{1-x}$As well,
**lattice-matched to InP** (so it can be grown without defects), emitting at 1550 nm.

In [ ]:
x_In = lattice_matched("InGaAs", "InP")
print(f"Lattice-matched composition: In{x_In:.3f}Ga{1-x_In:.3f}As")

Lws = np.arange(4, 16, 0.5)
lam = []
for Lw in Lws:
    s = Structure.single_well(well=("InGaAs", x_In), barrier="InP", Lw=Lw, Lb=20, substrate="InP")
    lam.append(Solver(s, T=300).solve(3, 3).interband()[0]["lam_nm"])
Lw_1550 = np.interp(1550, lam, Lws)
print(f"Well width for 1550 nm: {Lw_1550:.2f} nm")

plt.plot(Lws, lam, "s-"); plt.axhline(1550, color="r", ls="--")
plt.xlabel("Well width (nm)"); plt.ylabel("Wavelength (nm)")
plt.title("In$_{0.53}$Ga$_{0.47}$As/InP quantum well"); plt.grid(alpha=.3); plt.show()

s = Structure.single_well(well=("InGaAs", x_In), barrier="InP", Lw=round(Lw_1550, 1), Lb=20, substrate="InP")
s.describe()

*Note:* commercial 1.55 µm lasers often use compressively **strained** wells; strain is not yet
included in hetero1d, so this is the lattice-matched design.

## 3. Infrared photodetector (QWIP) for thermal imaging

**Problem:** quantum-well infrared photodetectors absorb in the 8–12 µm atmospheric window.
Best performance is obtained when the excited state sits **just below the barrier top**
("bound-to-quasibound" design): electrons excited by light escape easily, but dark current stays low.

**Method:** for each barrier composition, find the well width at which the second state just becomes bound.

In [ ]:
print(" x(Al)   Lw (nm)   lambda (um)   f_01")
design = []
for xb in (0.20, 0.22, 0.25, 0.28, 0.30):
    for Lw in np.arange(3.0, 9.0, 0.1):
        s = Structure.single_well(well="GaAs", barrier=("AlGaAs", xb), Lw=Lw, Lb=30)
        sol = Solver(s, T=77).solve(n_e=3, n_h=1)
        if sol.e_bound.sum() >= 2:                     # second state just bound
            t = sol.intersubband()[0]
            design.append((xb, Lw, t["lam_um"]))
            print(f" {xb:.2f}    {Lw:5.1f}      {t['lam_um']:6.2f}      {t['f']:.2f}")
            break
d = np.array(design)
plt.plot(d[:, 0], d[:, 2], "o-"); plt.axhspan(8, 12, color="orange", alpha=.2, label="8-12 µm window")
plt.xlabel("Al fraction in barrier"); plt.ylabel("Detection wavelength (µm)")
plt.title("Bound-to-quasibound QWIP design"); plt.legend(); plt.grid(alpha=.3); plt.show()

**Absorption of the 9 µm design versus doping** (absolute absorption per well, light polarised along z):

In [ ]:
xb, Lw = 0.22, 5.4
s = Structure.single_well(well="GaAs", barrier=("AlGaAs", xb), Lw=Lw, Lb=30)
sol = Solver(s, T=77).solve(n_e=3)
E = np.linspace(0.08, 0.2, 800)
for n2d in (1e11, 3e11, 5e11, 1e12):
    plt.plot(1.23984 / E, sol.isb_absorption(E, n2d_cm2=n2d, gamma=0.008) * 100, label=f"{n2d:.0e} cm$^{{-2}}$")
plt.xlabel("Wavelength (µm)"); plt.ylabel("Absorption per well (%)")
plt.title(f"QWIP: GaAs {Lw} nm / Al$_{{{xb}}}$GaAs, 77 K"); plt.legend(title="sheet doping"); plt.grid(alpha=.3); plt.show()

## 4. Electro-absorption modulator: quantum-confined Stark effect

**Problem:** optical modulators switch light on and off by applying a voltage. The electric field
pulls electrons and holes to opposite sides of the well, **red-shifting** the absorption edge
(QCSE). How big is the shift, and how much does the absorption change at a fixed wavelength?

In [ ]:
s = Structure.single_well(well="GaAs", barrier=("AlGaAs", 0.3), Lw=10, Lb=25)
sol = Solver(s, T=300)
sol.plot_stark("qcse.png", np.linspace(-80, 80, 33))
Image("qcse.png")

In [ ]:
E = np.linspace(1.40, 1.55, 600)
for F in (0, 40, 80):
    sv = Solver(s, T=300, field=F).solve(6, 6)
    plt.plot(E, sv.interband_absorption(E, gamma=0.004), label=f"{F} kV/cm")
E_op = 1.452
plt.axvline(E_op, color="k", ls=":", label=f"operating point {1239.84/E_op:.0f} nm")
plt.xlabel("Photon energy (eV)"); plt.ylabel("Absorption (arb.)")
plt.title("Absorption edge shifts with field: modulation"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 5. Resonant tunnelling diode (RTD)

**Problem:** RTDs are the fastest electronic oscillators (THz). The resonance width Γ sets the
electron lifetime τ = ħ/Γ in the well, which limits speed. How does barrier thickness control it?

In [ ]:
Ew = material("GaAs")["Ec"]
print(" barrier   E_res (meV)   width (ueV)   lifetime (ps)")
for Lb in (3, 4, 5, 6):
    rtd = Structure().add("GaAs", 20).add("AlGaAs", Lb, x=0.3).add("GaAs", 6) \
                     .add("AlGaAs", Lb, x=0.3).add("GaAs", 20)
    r = Solver(rtd, dz=0.05)
    Ec = np.linspace(Ew + 0.04, Ew + 0.09, 2001)
    Er = Ec[np.argmax(r.transmission(Ec))]
    Ef = np.linspace(Er - 0.004, Er + 0.004, 8001)
    Tf = r.transmission(Ef)
    half = Ef[Tf > Tf.max() / 2]
    G = half.max() - half.min()
    print(f"  {Lb} nm      {(Er-Ew)*1000:7.2f}      {G*1e6:9.1f}     {6.582e-16/G*1e12:7.2f}")
    if Lb in (3, 5):
        E = np.linspace(Ew + 0.001, Ew + 0.35, 3000)
        plt.semilogy((E - Ew) * 1000, np.maximum(r.transmission(E), 1e-12), label=f"barriers {Lb} nm")
plt.xlabel("Electron energy (meV)"); plt.ylabel("Transmission T(E)")
plt.title("Double-barrier RTD: GaAs/Al$_{0.3}$Ga$_{0.7}$As"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 6. Superlattice engineering: controlling the miniband

**Problem:** in a superlattice, wells couple into **minibands**. The miniband width controls
electron transport (Bloch oscillations, superlattice detectors, cascade injectors).

In [ ]:
widths = []
Lbs = [1, 1.5, 2, 3, 4, 5, 6]
for Lb in Lbs:
    sl = Structure.superlattice(10, well="GaAs", barrier=("AlGaAs", 0.3), Lw=5, Lb=Lb)
    q, d, bands = Solver(sl).minibands(np.linspace(Ew, Ew + 0.26, 5000))
    widths.append((bands[0][1] - bands[0][0]) * 1000)
    print(f"barrier {Lb} nm: miniband width {widths[-1]:.2f} meV")
plt.semilogy(Lbs, widths, "o-"); plt.xlabel("Barrier thickness (nm)"); plt.ylabel("Miniband width (meV)")
plt.title("GaAs 5 nm / Al$_{0.3}$Ga$_{0.7}$As superlattice"); plt.grid(alpha=.3, which="both"); plt.show()

sl = Structure.superlattice(10, well="GaAs", barrier=("AlGaAs", 0.3), Lw=5, Lb=2)
Solver(sl).solve(n_e=12).plot_summary("superlattice.png")
Image("superlattice.png")

## 7. HEMT: two-dimensional electron gas from modulation doping

**Problem:** high-electron-mobility transistors (mobile phones, satellite receivers, radio telescopes)
put donors in the AlGaAs and let the electrons move into an undoped GaAs channel, away from the
impurities. How much doping is right? Too much doping leaves electrons in the AlGaAs
("**parallel conduction**"), which ruins the mobility. This needs a **self-consistent
Schrödinger-Poisson** solution.

Structure (quantum-well HEMT): AlGaAs 30 nm | doped AlGaAs 10 nm | spacer 8 nm | **GaAs channel 15 nm** | AlGaAs 50 nm

In [ ]:
def hemt(Nd):
    h = Structure(name=f"QW-HEMT, Nd = {Nd:.0e}")
    h.add("AlGaAs", 30, x=0.3)                  # barrier
    h.add("AlGaAs", 10, x=0.3, doping=Nd)       # doped layer (donors, cm^-3)
    h.add("AlGaAs", 8, x=0.3)                   # undoped spacer
    h.add("GaAs", 15)                           # channel
    h.add("AlGaAs", 50, x=0.3)                  # back barrier
    return h

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for k, Nd in enumerate((6e17, 1e18)):
    sol = Solver(hemt(Nd), T=77).schrodinger_poisson()
    sol.solve(n_e=12)
    n2d = sol.Nd.sum() * sol.dz * 1e14
    EF, pops = sol.fermi_level(n2d)
    in_channel = sum(p * sol.layer_probabilities(sol.psi_e[:, i])[3] for i, p in enumerate(pops))
    print(f"Nd = {Nd:.0e} cm^-3:  n2D = {n2d:.1e} cm^-2,  electrons in channel = {in_channel/n2d*100:.1f} %")
    a = ax[k]
    a.plot(sol.z, sol.Ec, "k", label="Ec (self-consistent)")
    a.plot(sol.z, sol.Ec0, "k:", lw=0.8, label="Ec (no charge)")
    a.axhline(EF, color="r", ls="--", label="Fermi level")
    for i in range(3):
        if pops[i] > 0.01 * n2d:
            a.plot(sol.z, sol.Ee[i] + 0.06 * sol.psi_e[:, i]**2 / (sol.psi_e[:, i]**2).max(),
                   label=f"subband {i}: {pops[i]/n2d*100:.0f} %")
    a.set_ylim(sol.Ec.min() - 0.03, sol.Ec.min() + 0.35)
    a.set_xlabel("z (nm)"); a.set_ylabel("Energy (eV)"); a.legend(fontsize=8)
    a.set_title(("good design" if k == 0 else "over-doped: parallel conduction") + f", {n2d:.0e} cm$^{{-2}}$")
plt.tight_layout(); plt.show()

## 8. Magnetotransport: Landau levels, Shubnikov–de Haas, g-factor

**Problem:** in a magnetic field, electrons form Landau levels. When the Fermi level passes between
them the resistance oscillates (Shubnikov–de Haas) and, at low temperature, the quantum Hall
effect appears at integer filling factors ν. Where do these happen for a given sample?

In [ ]:
s = Structure.single_well(well="GaAs", barrier=("AlGaAs", 0.3), Lw=15, Lb=30)
n2d = 3e11
sol = Solver(s, T=1.5, Bz=5).solve()
print(f"electron g-factor in the well: {sol.g_factor(0):.3f}")
print(f"cyclotron energy at 5 T:       {sol.landau_levels()[0]['hwc']*1000:.2f} meV")
for nu in (1, 2, 4, 6):
    print(f"filling factor nu = {nu}:  B = {n2d / (2.417989e10 * nu):.2f} T")
sol.plot_magnetic("landau.png", np.linspace(0.4, 12, 300), n2d_cm2=n2d, gamma=0.0005)
Image("landau.png")

## 9. Double quantum well in a tilted magnetic field

**Problem:** coupled double wells are used in tunnelling spectroscopy, bilayer quantum Hall
physics and THz devices. An **in-plane** field shifts the two wells' dispersions to
k$_y$ = ±d/(2l$_B^2$), producing a "camel-back" lower band; the **perpendicular** component
creates Landau levels. A field tilted by θ gives both: B$_z$ = B cos θ, B$_\parallel$ = B sin θ.

In [ ]:
dqw = Structure.multi_well(2, well="GaAs", barrier=("AlGaAs", 0.3), Lw=15, Lb=2, Lb_out=30, name="DQW")
sol0 = Solver(dqw, T=1.5).solve(n_e=4)
print(f"tunnelling gap Delta_SAS = {(sol0.Ee[1]-sol0.Ee[0])*1000:.2f} meV")

ky = np.linspace(-0.25, 0.25, 161)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for Bp, ls in ((2, "-"), (5, "--"), (8, ":")):
    E = Solver(dqw, T=1.5, Bpar=Bp).solve(n_e=4).inplane_dispersion(ky, n_e=2)
    for n in range(2):
        ax[0].plot(ky, (E[:, n] - E.min()) * 1000, ls, color=f"C{n}", label=f"B$_\\parallel$={Bp} T" if n == 0 else None)
ax[0].set_ylim(0, 15); ax[0].set_xlabel("k$_y$ (1/nm)"); ax[0].set_ylabel("E (meV)")
ax[0].set_title("In-plane field: camel-back dispersion"); ax[0].legend()

B, angles, LL = 6.0, np.linspace(0, 80, 33), []
for th in angles:
    sv = Solver(dqw, T=1.5, Bz=B*np.cos(np.radians(th)), Bpar=B*np.sin(np.radians(th))).solve(n_e=4)
    LL.append([L["E"] for L in sv.landau_levels(B=sv.Bz, N_max=3)[:10]])
LL = (np.array(LL) - sol0.Ee[0]) * 1000
ax[1].plot(angles, LL, color="C0", lw=0.8)
ax[1].set_xlabel("Tilt angle θ (deg)"); ax[1].set_ylabel("Landau levels (meV)")
ax[1].set_title(f"Rotating a {B} T field"); plt.tight_layout(); plt.show()

## 10. Export results for your paper

Save band profile, wavefunctions and energies as CSV (Origin, Excel, MATLAB, gnuplot).

In [ ]:
s = Structure.single_well(well="GaAs", barrier=("AlGaAs", 0.3), Lw=10, Lb=25)
sol = Solver(s, T=300).solve(n_e=6, n_h=6)

data = np.column_stack([sol.z, sol.Ec, sol.Ev, sol.psi_e, sol.psi_hh])
header = "z_nm,Ec_eV,Ev_eV," + ",".join(f"psi_e{i}" for i in range(sol.psi_e.shape[1])) + "," + \
         ",".join(f"psi_hh{i}" for i in range(sol.psi_hh.shape[1]))
np.savetxt("wavefunctions.csv", data, delimiter=",", header=header, comments="")

E_meV = (sol.Ee - sol.Ec.min()) * 1000
np.savetxt("energies.csv", np.column_stack([np.arange(len(E_meV)), sol.Ee, E_meV, sol.e_bound]),
           delimiter=",", header="n,E_eV,E_above_well_meV,bound", comments="")
print(open("energies.csv").read())

try:                                  # download in Colab
    from google.colab import files
    files.download("wavefunctions.csv"); files.download("energies.csv")
except ImportError:
    print("Files saved in the current folder.")

## Model and limits
Single-band effective mass with position-dependent mass (BenDaniel-Duke), Γ valley,
no strain, no non-parabolicity; magnetic Landau levels for electrons with decoupled subbands.
Parameters: I. Vurgaftman, J. R. Meyer, L. R. Ram-Mohan, *J. Appl. Phys.* **89**, 5815 (2001).

**Cite / source code:** https://github.com/erw-del/hetero1d